# Dog Emotion Classifier

Audio-based classification of dog sounds using MFCCs and Machine Learning.

## Goal
Classify dog vocalizations (barking, whining, growling, howling) from audio recordings.

## Pipeline
1. Setup & imports
2. Audio visualization
3. Feature extraction (MFCCs)
4. Model training
5. Evaluation

In [40]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

In [41]:
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## 1. Load Audio Data

In [42]:
import os

DATA_PATH = "./audio"
CLASSES = ["barking", "whining", "growling", "howling"]

os.makedirs(DATA_PATH, exist_ok=True)

print(f"Classes: {CLASSES}")
print(f"Audio folder: {DATA_PATH}")


Classes: ['barking', 'whining', 'growling', 'howling']
Audio folder: ./audio


## 2. Visualize Audio

In [43]:
def plot_waveform(file_path, label):
    y, sr = librosa.load(file_path)

    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(y, sr=sr, color="#2c3e50")
    plt.title(f"Waveform: {label}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

## 3. Feature Extraction (MFCCs)

In [44]:
def extract_features(file_path):
    y, sr = librosa.load(file_path)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs, axis=1)
    return mfccs_mean

## 4. Load Data

In [51]:
DATA_PATH = "./ESC-50-master/audio"
META_PATH = "./ESC-50-master/meta/esc50.csv"
CLASSES = ["dog", "cat", "crow", "rooster"]

## 5. Build Dataset

In [52]:
import pandas as pd

def build_dataset(data_path, meta_path, classes):
    meta = pd.read_csv(meta_path)
    filtered = meta[meta["category"].isin(classes)]
    
    X = []
    y = []
    
    for _, row in filtered.iterrows():
        file_path = os.path.join(data_path, row["filename"])
        features = extract_features(file_path)
        if features is not None:
            X.append(features)
            y.append(row["category"])
    
    return np.array(X), np.array(y)

In [53]:
META_PATH = "./ESC-50-master/meta/esc50.csv"

X, y = build_dataset(DATA_PATH, META_PATH, CLASSES)

print(f"Samples: {X.shape[0]}")
print(f"Features per sample: {X.shape[1]}")
print(f"Classes: {np.unique(y)}")

Samples: 160
Features per sample: 13
Classes: ['cat' 'crow' 'dog' 'rooster']


## 6. Train / Test Split

In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Training samples: 128
Test samples: 32


In [55]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [56]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         cat       0.92      0.92      0.92        13
        crow       0.86      0.75      0.80         8
         dog       0.50      0.75      0.60         4
     rooster       0.67      0.57      0.62         7

    accuracy                           0.78        32
   macro avg       0.74      0.75      0.73        32
weighted avg       0.80      0.78      0.78        32



## 7. Train Model

In [48]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

model = RandomForestClassifier(n_estimators=100, random_state=42)

## 8. Evaluate Model

In [49]:
from sklearn.metrics import classification_report

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))